# Qwen2.5-Coder-14B-Instruct Inference API Server
Runs a FastAPI server on Kaggle and exposes it via ngrok.

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.3 MB/s eta 0:00:00:00:0100:01


In [2]:
# ── 2. Authenticate ngrok ────────────────────────────────────────────────────
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
# Option A: paste token directly.
# Option B: store as a Kaggle Secret named NGROK_TOKEN and uncomment below.
from kaggle_secrets import UserSecretsClient
NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
import os
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

In [ ]:
# ── 3. Load Qwen2.5-Coder-14B-Instruct ──────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"

import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret("HF_TOKEN")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically shards across both T4s
    torch_dtype=torch.float16,
)
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [ ]:
# ── 4. Define FastAPI app ────────────────────────────────────────────────────
import logging
import threading
import time
import traceback

import torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import Dict, List, Literal, Optional

# ── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
server_logger = logging.getLogger("inference_server")

def _gpu_mem_str() -> str:
    if not torch.cuda.is_available():
        return "no-cuda"
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    return f"alloc={allocated:.2f}GB reserved={reserved:.2f}GB"

app = FastAPI(title="Qwen2.5-Coder-14B-Instruct Inference API", version="1.0")

# ── Global exception handler — catches anything that escapes route handlers ───
@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    server_logger.error(
        "Unhandled exception on %s %s:\n%s",
        request.method, request.url.path,
        traceback.format_exc(),
    )
    return JSONResponse(status_code=500, content={"detail": str(exc)})

# ── Metrics ───────────────────────────────────────────────────────────────────
_metrics_lock = threading.Lock()
_metric_keys = ("input_tokens", "output_tokens", "requests")
_metrics = {}

def _empty_metrics():
    return {key: 0 for key in _metric_keys}

def _record_metrics(sid: str, input_toks: int, output_toks: int):
    session_id = sid or "default"
    with _metrics_lock:
        session_metrics = _metrics.setdefault(session_id, _empty_metrics())
        session_metrics["input_tokens"] += input_toks
        session_metrics["output_tokens"] += output_toks
        session_metrics["requests"] += 1

def _metrics_total():
    total = _empty_metrics()
    for session_metrics in _metrics.values():
        for key in _metric_keys:
            total[key] += session_metrics[key]
    return total


# ── Shared generation params ──────────────────────────────────────────────────
class GenParams(BaseModel):
    sid: Optional[str] = "default"
    max_new_tokens: Optional[int] = Field(default=512, ge=1, le=4096)
    temperature: Optional[float] = Field(default=0.7, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=0.95, ge=0.0, le=1.0)
    do_sample: Optional[bool] = True


# ── /chat ─────────────────────────────────────────────────────────────────────
class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str

class ChatRequest(GenParams):
    messages: List[Message]

class ChatResponse(BaseModel):
    role: str
    content: str
    num_new_tokens: int


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    t0 = time.time()
    server_logger.info(
        "[/chat] sid=%s max_new_tokens=%d do_sample=%s | GPU before: %s",
        req.sid, req.max_new_tokens, req.do_sample, _gpu_mem_str(),
    )
    try:
        text = tokenizer.apply_chat_template(
            [m.model_dump() for m in req.messages],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature if req.do_sample else 1.0,
                top_p=req.top_p if req.do_sample else 1.0,
                do_sample=req.do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_len:]
        reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
        _record_metrics(req.sid, input_len, len(new_tokens))

        elapsed = time.time() - t0
        server_logger.info(
            "[/chat] sid=%s OK | input_tokens=%d output_tokens=%d elapsed=%.1fs | GPU after: %s",
            req.sid, input_len, len(new_tokens), elapsed, _gpu_mem_str(),
        )
        return ChatResponse(role="assistant", content=reply, num_new_tokens=len(new_tokens))

    except Exception as e:
        elapsed = time.time() - t0
        server_logger.error(
            "[/chat] sid=%s FAILED after %.1fs | GPU: %s\n%s",
            req.sid, elapsed, _gpu_mem_str(), traceback.format_exc(),
        )
        raise HTTPException(status_code=500, detail=str(e))


# ── /generate ─────────────────────────────────────────────────────────────────
class GenerateRequest(GenParams):
    prompt: str

class GenerateResponse(BaseModel):
    prompt: str
    generated_text: str
    num_new_tokens: int


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    t0 = time.time()
    server_logger.info(
        "[/generate] sid=%s max_new_tokens=%d do_sample=%s | GPU before: %s",
        req.sid, req.max_new_tokens, req.do_sample, _gpu_mem_str(),
    )
    try:
        inputs = tokenizer(req.prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature if req.do_sample else 1.0,
                top_p=req.top_p if req.do_sample else 1.0,
                do_sample=req.do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_len:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        _record_metrics(req.sid, input_len, len(new_tokens))

        elapsed = time.time() - t0
        server_logger.info(
            "[/generate] sid=%s OK | input_tokens=%d output_tokens=%d elapsed=%.1fs | GPU after: %s",
            req.sid, input_len, len(new_tokens), elapsed, _gpu_mem_str(),
        )
        return GenerateResponse(prompt=req.prompt, generated_text=generated_text, num_new_tokens=len(new_tokens))

    except Exception as e:
        elapsed = time.time() - t0
        server_logger.error(
            "[/generate] sid=%s FAILED after %.1fs | GPU: %s\n%s",
            req.sid, elapsed, _gpu_mem_str(), traceback.format_exc(),
        )
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/metrics/{sid}")
def metrics_for_session(sid: str):
    with _metrics_lock:
        return dict(_metrics.get(sid, _empty_metrics()))


@app.get("/metrics")
def metrics():
    with _metrics_lock:
        return {"sessions": {sid: dict(v) for sid, v in _metrics.items()}, "total": _metrics_total()}


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "gpu": _gpu_mem_str()}


In [5]:
# ── 5. Start uvicorn in a background thread ──────────────────────────────────
import threading
import uvicorn

PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print(f"Uvicorn started on port {PORT}")

Uvicorn started on port 8000


In [6]:
# ── 6. Expose with ngrok ─────────────────────────────────────────────────────
import time
public_url = ""
time.sleep(2)  # give uvicorn a moment to bind
def expose_ngrok():
    global public_url
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    
    print("=" * 60)
    print(f"  Public URL : {public_url}")
    print(f"  Health     : {public_url}/health")
    print(f"  Docs       : {public_url}/docs")
    print(f"  Chat       : POST {public_url}/chat")
    print(f"  Generate   : POST {public_url}/generate")
    print("=" * 60)
ngrok_thread = threading.Thread(target=expose_ngrok, daemon=True)
ngrok_thread.start()

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [7]:
# ── 7. Quick smoke-test (/chat endpoint) ─────────────────────────────────────
import requests, json
while public_url == "":
    time.sleep(0.1)
resp = requests.post(
    f"{public_url}/chat",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user",   "content": "Write a Python function that computes the nth Fibonacci number."},
        ],
        "max_new_tokens": 256,
        "do_sample": False,
    },
    timeout=120,
)
print(json.dumps(resp.json(), indent=2))

  Public URL : https://cb17-34-170-63-74.ngrok-free.app
  Health     : https://cb17-34-170-63-74.ngrok-free.app/health
  Docs       : https://cb17-34-170-63-74.ngrok-free.app/docs
  Chat       : POST https://cb17-34-170-63-74.ngrok-free.app/chat
  Generate   : POST https://cb17-34-170-63-74.ngrok-free.app/generate


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     2001:ee0:4f84:1810:cda2:816c:2c96:adf1:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2001:ee0:4f84:1810:cda2:816c:2c96:adf1:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     34.170.63.74:0 - "POST /chat HTTP/1.1" 200 OK
{
  "role": "assistant",
  "content": "Here's a Python function to compute the nth Fibonacci number using an iterative approach:\n\n```python\ndef fibonacci(n):\n    # Base cases for the first two Fibonacci numbers\n    if n == 0:\n        return 0\n    elif n == 1:\n        return 1\n    \n    # Initialize the first two Fibonacci numbers\n    a, b = 0, 1\n    \n    # Compute the Fibonacci sequence iteratively up to the nth number\n    for _ in range(2, n + 1):\n        a, b = b, a + b\n    \n    # Return the nth Fibonacci number\n    return b\n```\n\nIn this solution, the function `fibonacci` calculates the nth Fibonacci number using an iterative method. The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, 

## Calling the API from your client

### `/chat` — recommended for instruct use
```python
import requests

PUBLIC_URL = "https://<your-ngrok-id>.ngrok-free.app"  # copy from cell 6

response = requests.post(
    f"{PUBLIC_URL}/chat",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user",   "content": "Implement quicksort in Python with comments."},
        ],
        "max_new_tokens": 512,
        "temperature": 0.7,
        "top_p": 0.95,
        "do_sample": True,
    },
    timeout=120,
)
print(response.json()["content"])
```

### `/generate` — raw completion (no chat template)
```python
response = requests.post(
    f"{PUBLIC_URL}/generate",
    json={"prompt": "def quicksort(arr):", "max_new_tokens": 256, "do_sample": False},
    timeout=120,
)
print(response.json()["generated_text"])
```

### Session metrics
```python
# Include sid in /chat and /generate requests to separate concurrent fuzzer instances.
response = requests.post(
    f"{PUBLIC_URL}/chat",
    json={
        "sid": "fuzzer-1",
        "messages": [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user", "content": "Implement quicksort in Python with comments."},
        ],
        "max_new_tokens": 512,
    },
    timeout=120,
)

# Per-session metrics.
response = requests.get(f"{PUBLIC_URL}/metrics/fuzzer-1")
print(response.json())
# {"input_tokens": 1234, "output_tokens": 5678, "requests": 42}

# All sessions plus aggregate totals.
response = requests.get(f"{PUBLIC_URL}/metrics")
print(response.json())
# {"sessions": {"fuzzer-1": {...}}, "total": {...}}
```

### Request schema
| Field | Type | Default | Description |
|---|---|---|---|
| `sid` | string | `default` | Fuzzer session id for per-instance metrics |
| `messages` | list of `{role, content}` | required (`/chat` only) | `role`: `system` / `user` / `assistant` |
| `prompt` | string | required (`/generate` only) | Raw text prompt |
| `max_new_tokens` | int | 512 | Max tokens to generate (1–4096) |
| `temperature` | float | 0.7 | Sampling temperature (0–2) |
| `top_p` | float | 0.95 | Nucleus sampling threshold |
| `do_sample` | bool | true | Greedy if false |

> **Kaggle GPU tip:** Qwen2.5-Coder-14B needs ~28 GB VRAM in float16.  
> Use **T4 x2** (2 × 16 GB) with `device_map="auto"`, or add `load_in_4bit=True` + `BitsAndBytesConfig` for a single T4.

In [ ]:
import time
from datetime import datetime

start = time.time()
MAX_RUNTIME = 120 * 60  # 70 minutes in seconds

while time.time() - start < MAX_RUNTIME:
    print("alive", datetime.now())
    time.sleep(300)

print("Stopping after 70 minutes")

alive 2026-05-28 03:58:56.630443
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
alive 2026-05-28 04:03:56.630728
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "GET /metrics HTTP/1.1" 200 OK
alive 2026-05-28 04:08:56.631163
alive 2026-05-28 04:13:56.631487
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     123.21.112.7:0 - "POST /chat HTTP/1.1" 200 OK
alive 2026-05-28 04:18:56.